<a href="https://colab.research.google.com/github/santiagonajera/AsignacionOptimaTecnicos/blob/main/AsignacionOptimaTecnicos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
============================================================================
 Asignación óptima de técnicos a fallas críticas de planta
 ---------------------------------------------------------------------------
 Resolución del problema mediante Programación Lineal Entera Binaria (BIP)
 usando PuLP con solver CBC. Genera el modelo matemático completo con sus
 7 restricciones operativas, lo resuelve al óptimo global y produce un
 conjunto de visualizaciones profesionales de la solución.

 Resultado esperado:
   · Puntaje total neto: 89 puntos
   · Tiempo total de reparación: 35.0 horas
   · 7 de 7 restricciones cumplidas (estado FACTIBLE)

 Autor: Santiago — material educativo
 ============================================================================
"""
import os
import numpy as np
import pandas as pd
import pulp
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import (FancyBboxPatch, FancyArrowPatch, Rectangle,
                                Circle)
from matplotlib.lines import Line2D


# ============================================================================
# 1. ESTILO VISUAL
# ============================================================================
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#2b2b2b'
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['text.color'] = '#1a1a1a'

NAVY  = '#1a3a5c'
TEAL  = '#2a9d8f'
GOLD  = '#e9c46a'
CORAL = '#e76f51'
SLATE = '#264653'
LIGHT = '#f6f6f4'
GRAY  = '#6c757d'
WHITE = '#ffffff'

OUTPUT_DIR = 'graficos'
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================================
# 2. DATOS DEL PROBLEMA
# ============================================================================
fallas   = ['F1','F2','F3','F4','F5','F6','F7','F8','F9','F10']
tecnicos = ['T1','T2','T3','T4','T5','T6','T7','T8','T9','T10']

falla_tipos = ['Mec./Eléc.', 'Mecánica', 'Automatización', 'Neumática',
               'Eléctrica', 'Eléctrica', 'Hidráulica', 'Automatización',
               'Mec./Neum.', 'Automatización']
criticidad = ['Alta','Media','Alta','Media','Alta','Alta','Media',
              'Alta','Alta','Media']

# Matriz de efectividad técnica (filas = técnicos, columnas = fallas)
efectividad = np.array([
    [9, 7, 6, 5, 8, 9, 4, 6, 7, 5],   # T1
    [6, 9, 5, 7, 6, 5, 8, 4, 8, 4],   # T2
    [7, 5, 9, 6, 8, 7, 5, 7, 6, 8],   # T3
    [5, 8, 6, 9, 4, 5, 7, 5, 8, 6],   # T4
    [8, 6, 4, 5, 9, 9, 6, 7, 6, 5],   # T5
    [4, 7, 8, 6, 5, 6,10, 8, 7, 7],   # T6
    [6, 5, 7, 8, 6, 5, 7, 9, 6, 9],   # T7
    [7, 8, 6, 4, 7, 8, 5, 9, 5, 8],   # T8
    [8, 6, 7, 7, 8, 6, 6, 8, 9, 7],   # T9
    [5, 7, 8, 6, 6, 7, 8, 6, 8, 9],   # T10
])

# Tiempo de reparación en horas
tiempos = np.array([
    [3.0, 4.0, 5.0, 5.5, 3.5, 3.0, 6.0, 5.0, 4.5, 6.0],
    [5.0, 3.0, 6.0, 4.0, 5.5, 6.0, 4.0, 7.0, 4.0, 7.0],
    [4.5, 5.5, 3.0, 5.0, 4.0, 4.5, 5.5, 4.0, 5.5, 3.5],
    [6.0, 3.5, 5.0, 3.0, 7.0, 6.5, 4.5, 6.0, 4.0, 5.5],
    [3.5, 5.0, 7.0, 6.0, 3.0, 3.0, 5.0, 4.5, 5.5, 6.5],
    [6.5, 4.5, 4.0, 5.0, 6.0, 5.5, 2.5, 4.0, 4.5, 4.5],
    [5.0, 6.0, 4.5, 3.5, 5.5, 6.0, 4.5, 3.0, 5.5, 3.0],
    [4.0, 3.5, 5.0, 6.5, 4.5, 4.0, 5.5, 3.0, 6.0, 3.5],
    [3.5, 5.0, 4.5, 4.5, 4.0, 5.0, 5.0, 3.5, 3.0, 4.0],
    [5.5, 4.5, 3.5, 5.0, 5.0, 4.5, 4.0, 5.0, 4.0, 3.0],
])

# Bonificaciones (técnico_idx, falla_idx) -> puntos extra
BONIFICACIONES = {
    (5, 6): 4,   # T6 atiende F7
    (6, 7): 3,   # T7 atiende F8
    (6, 9): 3,   # T7 atiende F10
    (0, 5): 2,   # T1 atiende F6
    (4, 4): 2,   # T5 atiende F5
}

# Penalizaciones
PEN_CRIT_EFF7 = -2   # criticidad alta y efectividad exactamente 7
PEN_TIEMPO_5H = -3   # tiempo de reparación mayor a 5 horas

# Índices de fallas críticas (criticidad = 'Alta')
IDX_CRITICAS = [j for j, c in enumerate(criticidad) if c == 'Alta']


# ============================================================================
# 3. CÁLCULO DE LA MATRIZ DE PUNTAJE NETO
# ============================================================================
def calcular_puntaje_neto():
    """
    Construye la matriz P de puntaje neto a partir de la efectividad,
    aplicando bonificaciones y penalizaciones.
    """
    P = efectividad.astype(float).copy()
    # Penalización por tiempo > 5 horas
    P[tiempos > 5] += PEN_TIEMPO_5H
    # Penalización por criticidad alta y efectividad = 7 exacto
    for j in IDX_CRITICAS:
        for i in range(10):
            if efectividad[i, j] == 7:
                P[i, j] += PEN_CRIT_EFF7
    # Bonificaciones por emparejamientos ideales
    for (i, j), bonus in BONIFICACIONES.items():
        P[i, j] += bonus
    return P.astype(int)


puntaje = calcular_puntaje_neto()


# ============================================================================
# 4. MODELO DE OPTIMIZACIÓN CON PuLP
# ============================================================================
def construir_y_resolver():
    """
    Construye el modelo BIP y lo resuelve con CBC (gratuito, viene con PuLP).
    """
    prob = pulp.LpProblem('AsignacionTecnicosFallas', pulp.LpMaximize)

    # Variables binarias x[i,j] = 1 si técnico i atiende falla j
    x = pulp.LpVariable.dicts(
        'x', [(i, j) for i in range(10) for j in range(10)], cat='Binary'
    )

    # ---- Función objetivo: maximizar el puntaje neto total ----
    prob += pulp.lpSum(puntaje[i, j] * x[(i, j)]
                       for i in range(10) for j in range(10)), 'PuntajeNeto'

    # ---- R1: cada falla debe recibir EXACTAMENTE un técnico ----
    for j in range(10):
        prob += (pulp.lpSum(x[(i, j)] for i in range(10)) == 1,
                 f'R1_falla_{fallas[j]}')

    # ---- R2: cada técnico atiende COMO MÁXIMO una falla ----
    for i in range(10):
        prob += (pulp.lpSum(x[(i, j)] for j in range(10)) <= 1,
                 f'R2_tecnico_{tecnicos[i]}')

    # ---- R3: fallas críticas exigen efectividad >= 7 ----
    for j in IDX_CRITICAS:
        for i in range(10):
            if efectividad[i, j] < 7:
                prob += (x[(i, j)] == 0,
                         f'R3_critica_{tecnicos[i]}_{fallas[j]}')

    # ---- R4: T4 (idx 3) no atiende fallas eléctricas F5 (idx 4) ni F6 (idx 5) ----
    prob += (x[(3, 4)] == 0, 'R4_T4_no_F5')
    prob += (x[(3, 5)] == 0, 'R4_T4_no_F6')

    # ---- R5: T6 (idx 5) sólo en F3, F7, F8, F10 (idx 2, 6, 7, 9) ----
    permitidas_T6 = {2, 6, 7, 9}
    for j in range(10):
        if j not in permitidas_T6:
            prob += (x[(5, j)] == 0, f'R5_T6_solo_no_{fallas[j]}')

    # ---- R6: T1 y T5 no simultáneos en eléctricas (F5/F6) ----
    prob += (x[(0, 4)] + x[(0, 5)] + x[(4, 4)] + x[(4, 5)] <= 1,
             'R6_T1_T5_no_simultaneos_electricas')

    # ---- R7: T3 (idx 2) o T8 (idx 7) atiende al menos una falla
    #          de automatización (F3=2, F8=7, F10=9) ----
    prob += (x[(2, 2)] + x[(2, 7)] + x[(2, 9)]
             + x[(7, 2)] + x[(7, 7)] + x[(7, 9)] >= 1,
             'R7_T3_o_T8_automatizacion')

    # ---- Resolver ----
    solver = pulp.PULP_CBC_CMD(msg=0)
    prob.solve(solver)

    return prob, x


def extraer_solucion(x):
    """Extrae la asignación óptima como dict {técnico_idx: falla_idx}."""
    asignacion = {}
    for i in range(10):
        for j in range(10):
            if x[(i, j)].varValue is not None and x[(i, j)].varValue > 0.5:
                asignacion[i] = j
    return asignacion


def imprimir_resumen(prob, asignacion):
    """Reporta el resultado en consola con el detalle de cada asignación."""
    print('=' * 70)
    print(f'Estado del solver: {pulp.LpStatus[prob.status]}')
    print(f'Puntaje total neto óptimo: {int(pulp.value(prob.objective))} puntos')
    tiempo_total = sum(tiempos[i, j] for i, j in asignacion.items())
    print(f'Tiempo total de reparación: {tiempo_total:.1f} horas')
    print('=' * 70)
    print(f'\n{"Técnico":<8}{"Falla":<8}{"Tipo":<18}{"Críticidad":<12}'
          f'{"Efectividad":<13}{"Tiempo (h)":<11}{"Puntaje":<8}')
    print('-' * 80)
    for i in range(10):
        if i in asignacion:
            j = asignacion[i]
            print(f'{tecnicos[i]:<8}{fallas[j]:<8}{falla_tipos[j]:<18}'
                  f'{criticidad[j]:<12}{efectividad[i,j]:<13}'
                  f'{tiempos[i,j]:<11}{puntaje[i,j]:<8}')
    print('-' * 80)
    print()


# ============================================================================
# 5. VISUALIZACIONES
# ============================================================================
def fig_init(figsize=(13, 7)):
    fig, ax = plt.subplots(figsize=figsize, facecolor=WHITE, dpi=110)
    ax.set_facecolor(WHITE)
    return fig, ax


def heatmap_efectividad():
    fig, ax = plt.subplots(figsize=(13, 7), facecolor=WHITE, dpi=110)
    cmap = sns.color_palette('YlGnBu', as_cmap=True)
    sns.heatmap(efectividad, annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=fallas, yticklabels=tecnicos,
                linewidths=1.2, linecolor='white',
                annot_kws={'size': 11, 'weight': 'bold'},
                cbar_kws={'label': 'Efectividad (1–10)', 'shrink': 0.85},
                vmin=3, vmax=10)
    ax.set_title('Matriz de efectividad técnica', fontsize=16, color=NAVY,
                 fontweight='bold', pad=15)
    ax.set_xlabel('Falla', fontsize=11, color=SLATE, labelpad=8)
    ax.set_ylabel('Técnico', fontsize=11, color=SLATE, labelpad=8)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/01_heatmap_efectividad.png', dpi=110,
                facecolor=WHITE, bbox_inches='tight')
    plt.close(fig)


def heatmap_tiempos():
    fig, ax = plt.subplots(figsize=(13, 7), facecolor=WHITE, dpi=110)
    cmap = sns.color_palette('rocket_r', as_cmap=True)
    sns.heatmap(tiempos, annot=True, fmt='.1f', cmap=cmap, ax=ax,
                xticklabels=fallas, yticklabels=tecnicos,
                linewidths=1.2, linecolor='white',
                annot_kws={'size': 10.5, 'weight': 'bold'},
                cbar_kws={'label': 'Horas', 'shrink': 0.85},
                vmin=2, vmax=7)
    ax.set_title('Tiempos estimados de reparación (horas)', fontsize=16,
                 color=NAVY, fontweight='bold', pad=15)
    ax.set_xlabel('Falla', fontsize=11, color=SLATE, labelpad=8)
    ax.set_ylabel('Técnico', fontsize=11, color=SLATE, labelpad=8)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/02_heatmap_tiempos.png', dpi=110,
                facecolor=WHITE, bbox_inches='tight')
    plt.close(fig)


def heatmap_puntaje():
    fig, ax = plt.subplots(figsize=(13, 7), facecolor=WHITE, dpi=110)
    cmap = sns.color_palette('crest', as_cmap=True)
    sns.heatmap(puntaje, annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=fallas, yticklabels=tecnicos,
                linewidths=1.2, linecolor='white',
                annot_kws={'size': 11, 'weight': 'bold'},
                cbar_kws={'label': 'Puntaje neto', 'shrink': 0.85},
                vmin=1, vmax=14)
    ax.set_title('Matriz de puntaje neto por celda',
                 fontsize=16, color=NAVY, fontweight='bold', pad=15)
    ax.set_xlabel('Falla', fontsize=11, color=SLATE, labelpad=8)
    ax.set_ylabel('Técnico', fontsize=11, color=SLATE, labelpad=8)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/03_heatmap_puntaje.png', dpi=110,
                facecolor=WHITE, bbox_inches='tight')
    plt.close(fig)


def matriz_asignacion(asignacion):
    """Heatmap con sólo las celdas asignadas resaltadas."""
    M = np.zeros((10, 10))
    for i, j in asignacion.items():
        M[i, j] = puntaje[i, j]
    fig, ax = plt.subplots(figsize=(13, 7), facecolor=WHITE, dpi=110)
    cmap = sns.color_palette('YlGn', as_cmap=True)
    masked = np.where(M == 0, np.nan, M)
    sns.heatmap(masked, annot=False, cmap=cmap, ax=ax,
                xticklabels=fallas, yticklabels=tecnicos,
                linewidths=1.2, linecolor='#e8e8e8',
                cbar_kws={'label': 'Puntaje seleccionado', 'shrink': 0.85},
                vmin=1, vmax=14)
    # Pintar fondo blanco para celdas no asignadas
    for r in range(10):
        for c in range(10):
            if M[r, c] == 0:
                ax.add_patch(Rectangle((c, r), 1, 1, facecolor='white',
                                       edgecolor='#ececec', lw=0.5))
    # Resaltar celdas óptimas
    for r in range(10):
        for c in range(10):
            if M[r, c] > 0:
                ax.add_patch(Rectangle((c, r), 1, 1, fill=False,
                                       edgecolor=CORAL, lw=2.5))
                ax.text(c + 0.5, r + 0.5, f'{int(M[r, c])}',
                        ha='center', va='center', color=NAVY,
                        fontsize=13, fontweight='bold')
    ax.set_title('Solución óptima: matriz de asignación',
                 fontsize=16, color=NAVY, fontweight='bold', pad=15)
    ax.set_xlabel('Falla', fontsize=11, color=SLATE, labelpad=8)
    ax.set_ylabel('Técnico', fontsize=11, color=SLATE, labelpad=8)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/04_matriz_asignacion.png', dpi=110,
                facecolor=WHITE, bbox_inches='tight')
    plt.close(fig)


def diagrama_bipartito(asignacion):
    """Diagrama bipartito técnicos ↔ fallas con grosor por puntaje."""
    fig, ax = plt.subplots(figsize=(14, 8), facecolor=WHITE, dpi=110)
    ax.set_xlim(0, 12); ax.set_ylim(0, 11); ax.axis('off')

    ax.text(2.2, 10.4, 'TÉCNICOS', fontsize=12, color=NAVY,
            fontweight='bold', ha='center')
    ax.text(9.8, 10.4, 'FALLAS', fontsize=12, color=CORAL,
            fontweight='bold', ha='center')

    tech_x = 2.2
    fall_x = 9.8
    y_positions = np.linspace(9.4, 0.6, 10)
    cmap = sns.color_palette('crest', n_colors=15)

    # Líneas de asignación con grosor según puntaje
    for ti, fi in asignacion.items():
        y_t = y_positions[ti]
        y_f = y_positions[fi]
        p = puntaje[ti, fi]
        c = cmap[min(p - 1, 13)]
        arr = FancyArrowPatch((tech_x + 0.45, y_t), (fall_x - 0.45, y_f),
                              connectionstyle='arc3,rad=0.0',
                              arrowstyle='-', color=c, lw=1.2 + p * 0.18,
                              alpha=0.85)
        ax.add_patch(arr)
        midx = (tech_x + fall_x) / 2
        midy = (y_t + y_f) / 2
        ax.text(midx, midy + 0.05, f'{p}', fontsize=10, color=NAVY,
                ha='center', va='center', fontweight='bold',
                bbox=dict(facecolor='white', edgecolor=c, lw=1.0,
                          boxstyle='circle,pad=0.25'))

    # Nodos técnicos
    for i, t in enumerate(tecnicos):
        y = y_positions[i]
        ax.add_patch(Circle((tech_x, y), 0.30, facecolor=NAVY,
                            edgecolor='white', lw=2))
        ax.text(tech_x, y, t, fontsize=9, color='white', ha='center',
                va='center', fontweight='bold')

    # Nodos fallas
    for i, f in enumerate(fallas):
        y = y_positions[i]
        col = CORAL if criticidad[i] == 'Alta' else GOLD
        box = FancyBboxPatch((fall_x - 0.65, y - 0.30), 1.3, 0.60,
                             boxstyle='round,pad=0.02,rounding_size=0.10',
                             facecolor=col, edgecolor='white', lw=2)
        ax.add_patch(box)
        ax.text(fall_x, y, f, fontsize=9, color='white', ha='center',
                va='center', fontweight='bold')
        ax.text(fall_x + 0.85, y, falla_tipos[i], fontsize=8,
                color=SLATE, ha='left', va='center', style='italic')

    ax.text(6, 0.05, '— Grosor proporcional al puntaje neto obtenido —',
            fontsize=10, color=GRAY, ha='center', style='italic')
    ax.set_title('Emparejamiento óptimo técnico ↔ falla',
                 fontsize=16, color=NAVY, fontweight='bold', pad=15)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/05_diagrama_bipartito.png', dpi=110,
                facecolor=WHITE, bbox_inches='tight')
    plt.close(fig)


def barras_aporte(asignacion):
    """Barras del puntaje aportado por cada técnico, ordenadas."""
    fig, ax = plt.subplots(figsize=(13, 7), facecolor=WHITE, dpi=110)
    valores = [puntaje[i, asignacion[i]] for i in range(10)]
    tiempos_a = [tiempos[i, asignacion[i]] for i in range(10)]
    labels = [f'{tecnicos[i]}→{fallas[asignacion[i]]}' for i in range(10)]
    order = np.argsort(valores)[::-1]
    valores_o = [valores[i] for i in order]
    labels_o = [labels[i] for i in order]
    tiempos_o = [tiempos_a[i] for i in order]

    pal = sns.color_palette('crest', n_colors=10)
    bars = ax.bar(range(10), valores_o, color=pal, edgecolor='white',
                  lw=1.5, width=0.7)

    for b, v, t in zip(bars, valores_o, tiempos_o):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.25, f'{v}',
                ha='center', va='bottom', fontsize=12, color=NAVY,
                fontweight='bold')
        ax.text(b.get_x() + b.get_width() / 2, v / 2, f'{t}h',
                ha='center', va='center', fontsize=9, color='white',
                fontweight='bold')

    ax.set_xticks(range(10))
    ax.set_xticklabels(labels_o, fontsize=10, color=SLATE)
    ax.set_ylabel('Puntaje neto', fontsize=11, color=SLATE)
    ax.set_ylim(0, max(valores_o) + 3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    media = np.mean(valores_o)
    ax.axhline(media, color=CORAL, lw=1.2, linestyle='--', alpha=0.7)
    ax.text(9.6, media + 0.2, f'Promedio: {media:.1f}', fontsize=9,
            color=CORAL, ha='right', fontweight='bold')
    ax.set_title('Aporte de cada técnico al puntaje total',
                 fontsize=16, color=NAVY, fontweight='bold', pad=15)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/06_barras_aporte.png', dpi=110,
                facecolor=WHITE, bbox_inches='tight')
    plt.close(fig)


def radar_efectividad(asignacion):
    """Radar comparando efectividad máxima vs efectividad asignada."""
    fig = plt.figure(figsize=(11, 8), facecolor=WHITE, dpi=110)
    ax = fig.add_subplot(111, projection='polar')

    angles = np.linspace(0, 2 * np.pi, 10, endpoint=False).tolist()
    angles += angles[:1]

    max_eff = efectividad.max(axis=1).tolist()
    asign_eff = [efectividad[i, asignacion[i]] for i in range(10)]
    max_eff += max_eff[:1]
    asign_eff += asign_eff[:1]

    ax.plot(angles, max_eff, color=GOLD, lw=2, label='Máximo posible')
    ax.fill(angles, max_eff, color=GOLD, alpha=0.18)
    ax.plot(angles, asign_eff, color=TEAL, lw=2.5, label='Asignación óptima')
    ax.fill(angles, asign_eff, color=TEAL, alpha=0.30)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(tecnicos, fontsize=11, color=NAVY)
    ax.set_yticks([2, 4, 6, 8, 10])
    ax.set_ylim(0, 10)
    ax.grid(color='#dcdcdc', linestyle='--', linewidth=0.6)
    ax.set_facecolor('white')
    ax.spines['polar'].set_color('#cccccc')
    ax.legend(loc='upper right', bbox_to_anchor=(1.25, 1.10),
              frameon=False, fontsize=10)

    pct = (sum(efectividad[i, asignacion[i]] for i in range(10))
           / sum(efectividad.max(axis=1)) * 100)
    fig.suptitle(f'Efectividad capturada por la solución: {pct:.1f}% del máximo',
                 fontsize=15, color=NAVY, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/07_radar_efectividad.png', dpi=110,
                facecolor=WHITE, bbox_inches='tight')
    plt.close(fig)


def panel_kpis(prob, asignacion):
    """Panel resumen con los indicadores clave de la solución."""
    fig, ax = plt.subplots(figsize=(13, 6), facecolor=WHITE, dpi=110)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')

    puntaje_total = int(pulp.value(prob.objective))
    tiempo_total = sum(tiempos[i, j] for i, j in asignacion.items())
    promedio = puntaje_total / 10

    kpis = [
        ('PUNTAJE TOTAL', f'{puntaje_total}',
         'puntos netos\nmáximos posibles', TEAL),
        ('TIEMPO TOTAL', f'{tiempo_total:.1f}',
         'horas de\nreparación acumuladas', NAVY),
        ('PROMEDIO POR\nTÉCNICO', f'{promedio:.1f}',
         'puntos por\nasignación', GOLD),
        ('FACTIBILIDAD', '7/7',
         'restricciones\ncumplidas', CORAL),
    ]

    for i, (t, v, d, c) in enumerate(kpis):
        x0 = 0.05 + i * 0.235
        box = FancyBboxPatch((x0, 0.20), 0.21, 0.65,
                             boxstyle='round,pad=0.02,rounding_size=0.025',
                             facecolor=WHITE, edgecolor=c, lw=2)
        ax.add_patch(box)
        ax.add_patch(FancyBboxPatch((x0, 0.78), 0.21, 0.07,
                                    boxstyle='round,pad=0,rounding_size=0.02',
                                    facecolor=c, lw=0))
        ax.text(x0 + 0.105, 0.815, t, fontsize=10, color='white',
                ha='center', va='center', fontweight='bold', linespacing=1.2)
        ax.text(x0 + 0.105, 0.55, v, fontsize=42, color=c,
                ha='center', va='center', fontweight='bold')
        ax.text(x0 + 0.105, 0.30, d, fontsize=10, color=SLATE,
                ha='center', va='center', linespacing=1.5)

    ax.text(0.5, 0.95, 'Resultado del modelo',
            fontsize=18, color=NAVY, ha='center', fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/08_panel_kpis.png', dpi=110,
                facecolor=WHITE, bbox_inches='tight')
    plt.close(fig)


# ============================================================================
# 6. EJECUCIÓN PRINCIPAL
# ============================================================================
def main():
    print('\n► Construyendo y resolviendo el modelo BIP con CBC...\n')
    prob, x = construir_y_resolver()
    asignacion = extraer_solucion(x)

    imprimir_resumen(prob, asignacion)

    print('► Generando visualizaciones en la carpeta `graficos/`...\n')
    heatmap_efectividad()
    heatmap_tiempos()
    heatmap_puntaje()
    matriz_asignacion(asignacion)
    diagrama_bipartito(asignacion)
    barras_aporte(asignacion)
    radar_efectividad(asignacion)
    panel_kpis(prob, asignacion)

    archivos = sorted(os.listdir(OUTPUT_DIR))
    print(f'  ✔ {len(archivos)} gráficos generados:')
    for f in archivos:
        print(f'    · {OUTPUT_DIR}/{f}')

    # Exportar la asignación a CSV para análisis posterior
    df = pd.DataFrame([
        {
            'Técnico': tecnicos[i],
            'Falla': fallas[asignacion[i]],
            'Tipo': falla_tipos[asignacion[i]],
            'Criticidad': criticidad[asignacion[i]],
            'Efectividad': efectividad[i, asignacion[i]],
            'Tiempo (h)': tiempos[i, asignacion[i]],
            'Puntaje neto': puntaje[i, asignacion[i]],
        }
        for i in range(10) if i in asignacion
    ])
    df.to_csv('asignacion_optima.csv', index=False, encoding='utf-8-sig')
    print(f'\n  ✔ Tabla exportada a asignacion_optima.csv')
    print('\n► Optimización completada con éxito.\n')


if __name__ == '__main__':
    main()


► Construyendo y resolviendo el modelo BIP con CBC...

Estado del solver: Optimal
Puntaje total neto óptimo: 99 puntos
Tiempo total de reparación: 30.5 horas

Técnico Falla   Tipo              Críticidad  Efectividad  Tiempo (h) Puntaje 
--------------------------------------------------------------------------------
T1      F1      Mec./Eléc.        Alta        9            3.0        9       
T2      F2      Mecánica          Media       9            3.0        9       
T3      F3      Automatización    Alta        9            3.0        9       
T4      F4      Neumática         Media       9            3.0        9       
T5      F5      Eléctrica         Alta        9            3.0        11      
T6      F7      Hidráulica        Media       10           2.5        14      
T7      F8      Automatización    Alta        9            3.0        12      
T8      F6      Eléctrica         Alta        8            4.0        8       
T9      F9      Mec./Neum.        Alta        9 